In [1]:
import cv2
import os

labels = []
def load_images_from_folder(folder):
    data = []   
    for filename in os.listdir(folder):
        if filename.lower().endswith('.jpg'):
            img = cv2.imread(os.path.join(folder, filename))
            data.append(img)

            # for printing results
            labels.append(filename)
    return data

dog_images = load_images_from_folder('training/dog')
cat_images = load_images_from_folder('training/cat')

In [2]:
import numpy as np

dog_images = np.array(dog_images)
cat_images = np.array(cat_images)

print(dog_images.shape)
print(cat_images.shape)

(10, 224, 224, 3)
(10, 224, 224, 3)


In [4]:
# use it as a magic tool
from transformers import CLIPProcessor, CLIPModel
import torch

clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

# CLIP
def magic_function(images):
    inputs = clip_processor(images=images, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = clip_model.get_image_features(**inputs)
        
        # Safely extract tensor & normalize
        if hasattr(outputs, 'image_embeds'):
             features = outputs.image_embeds
        elif hasattr(outputs, 'pooler_output'):
             features = outputs.pooler_output
        else:
             features = outputs

    # Normalize features (recommended for CLIP so cosine similarity works well)
    return (features / features.norm(p=2, dim=-1, keepdim=True)).cpu().numpy()

dog_images_features = magic_function(list(dog_images))
cat_images_features = magic_function(list(cat_images))

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 1691.51it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
print(dog_images_features.shape)
print(cat_images_features.shape)

(10, 512)
(10, 512)


In [6]:
merged_features = np.concatenate((dog_images_features, cat_images_features), axis=0)
print(merged_features.shape)

(20, 512)


In [7]:

from sklearn.cluster import KMeans

# KMeans
kmeans = KMeans(n_clusters=2, random_state=0).fit(merged_features)
print(kmeans.labels_)

for real_label, predicted_label in zip(labels, kmeans.labels_):
    print(f"Cluster {predicted_label}: {real_label}")

[1 1 1 1 1 1 1 0 1 1 0 0 0 0 0 0 0 0 0 0]
Cluster 1: dog.0.jpg
Cluster 1: dog.1.jpg
Cluster 1: dog.2.jpg
Cluster 1: dog.3.jpg
Cluster 1: dog.4.jpg
Cluster 1: dog.5.jpg
Cluster 1: dog.6.jpg
Cluster 0: dog.7.jpg
Cluster 1: dog.8.jpg
Cluster 1: dog.9.jpg
Cluster 0: cat.0.jpg
Cluster 0: cat.1.jpg
Cluster 0: cat.2.jpg
Cluster 0: cat.3.jpg
Cluster 0: cat.4.jpg
Cluster 0: cat.5.jpg
Cluster 0: cat.6.jpg
Cluster 0: cat.7.jpg
Cluster 0: cat.8.jpg
Cluster 0: cat.9.jpg
